# 02b — Điều tra domain gap: `tomato_ripeness_v1` trên `test_outdomain_openfield`

**Bối cảnh:** `02_train_ripeness_baseline.ipynb` cho thấy mAP@0.5 rơi từ 0,878 (`test` cùng miền) xuống 0,473 (`test_outdomain_openfield`) — chênh lệch 0,405, vượt xa ngưỡng cảnh báo 0,15. `fruit_turning` sụp mạnh nhất (AP50 0,823 → 0,260).

**Câu hỏi cần trả lời:** domain gap này là do (a) **lỗi nhãn** trong `openfield_bd` — giống trường hợp `leaf_mold` đã tìm thấy ở nhánh bệnh lá, hay (b) **domain shift thật** (camera/ánh sáng/môi trường ngoài đồng khác hẳn nhà kính) mà việc gắn lại nhãn không giải quyết được?

Lưu ý khác biệt quan trọng so với nhánh bệnh lá: class mapping của `openfield_bd` đã được xác nhận qua `data.yaml` **thật** (`names: ['full_ripened', 'half_ripened', 'green']`) từ `01_build_tomato_ripeness_v1.ipynb`, không phải giả định — nên khả năng cao đây là domain shift thật hơn là lỗi map class. Notebook này kiểm chứng bằng dữ liệu thay vì đoán.

## Cách điều tra
1. So khớp dự đoán của `best.pt` với ground truth trên từng ảnh `test_outdomain_openfield` bằng IoU — phân loại mỗi box GT thành đúng / nhầm class / bị bỏ sót (miss), và mỗi box dự đoán thừa thành báo động giả.
2. Ma trận nhầm lẫn thật (true class → predicted class) để biết model nhầm theo hướng nào (vd có phải toàn bộ đổ dồn về `fruit_green_unripe` không).
3. Xem trực quan top ảnh sai nhiều nhất — kèm cả box GT lẫn box dự đoán để tự đánh giá: ảnh gắn nhãn đúng nhưng model đoán sai (domain shift), hay nhãn GT trông đáng ngờ (lỗi nhãn)?
4. So sánh thống kê ảnh (độ sáng, độ bão hòa màu) giữa nguồn trong miền (Laboro/AgRob/PlantFactory) và `openfield_bd` — định lượng mức độ khác biệt môi trường chụp.

## Trước khi chạy
1. **Add Input** → 2 Kaggle Dataset: output đã Save Version của `01_build_tomato_ripeness_v1.ipynb` (`tomato_ripeness_v1`) và output đã Save Version của `02_train_ripeness_baseline.ipynb` (chứa `best.pt`).
2. Accelerator: GPU giúp nhanh hơn nhưng không bắt buộc (chỉ predict trên 604 ảnh, không train).
3. Không cần Internet ngoài `pip install ultralytics`.

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=False)

import random
from collections import defaultdict
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image

random.seed(42)
np.random.seed(42)

print("CUDA khả dụng:", torch.cuda.is_available())

### Bước 0 — Tự nhận diện dataset + model input

In [ ]:
TARGET_CLASSES = ["fruit_green_unripe", "fruit_turning", "fruit_ripe"]


def find_dataset_root():
    for yf in Path("/kaggle/input").rglob("data.yaml"):
        try:
            y = yaml.safe_load(yf.read_text(encoding="utf-8"))
        except Exception:
            continue
        names = y.get("names") if isinstance(y, dict) else None
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names == TARGET_CLASSES:
            return yf.parent
    return None


def find_model_path():
    hits = list(Path("/kaggle/input").rglob("tomato_ripeness_yolov8n_baseline.pt"))
    return hits[0] if hits else None


DATASET_ROOT = find_dataset_root()
MODEL_PATH = find_model_path()

if DATASET_ROOT is None:
    print("Các thư mục cấp 1 trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*")):
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy data.yaml khớp 3 class của tomato_ripeness_v1. "
        "Kiểm tra đã Add Input output của 01_build_tomato_ripeness_v1.ipynb chưa."
    )
if MODEL_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy tomato_ripeness_yolov8n_baseline.pt trong /kaggle/input. "
        "Kiểm tra đã Add Input output đã Save Version của 02_train_ripeness_baseline.ipynb chưa "
        "(dataset đó phải chứa thư mục models/)."
    )

OUTDOMAIN_DIR = DATASET_ROOT / "test_outdomain_openfield"
if not OUTDOMAIN_DIR.exists():
    raise FileNotFoundError(f"Không thấy {OUTDOMAIN_DIR} — kiểm tra lại dataset input.")

print("DATASET_ROOT :", DATASET_ROOT)
print("MODEL_PATH   :", MODEL_PATH)
print("OUTDOMAIN_DIR:", OUTDOMAIN_DIR, "-", len(list((OUTDOMAIN_DIR / "images").glob("*"))), "ảnh")

### Bước 1 — So khớp dự đoán với ground truth theo IoU
Với mỗi ảnh: so từng box GT với box dự đoán có IoU cao nhất (ngưỡng 0.5, không phân biệt class khi so khớp vị trí — để phát hiện đúng trường hợp "định vị đúng nhưng phân loại sai"). Mỗi box GT thành 1 trong 3 loại: `correct` (đúng cả vị trí lẫn class), `misclassified` (đúng vị trí, sai class), `missed` (không có box dự đoán nào khớp). Box dự đoán không khớp GT nào → `extra` (báo động giả).

In [ ]:
from ultralytics import YOLO


def yolo_txt_to_xyxy(label_path, w, h):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        xmin, ymin = (xc - bw / 2) * w, (yc - bh / 2) * h
        xmax, ymax = (xc + bw / 2) * w, (yc + bh / 2) * h
        boxes.append((cid, xmin, ymin, xmax, ymax))
    return boxes


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


model = YOLO(str(MODEL_PATH))

IOU_THRESH = 0.5
images_dir = OUTDOMAIN_DIR / "images"
labels_dir = OUTDOMAIN_DIR / "labels"
all_imgs = sorted(images_dir.glob("*"))

per_image_records = []
confusion_pairs = defaultdict(int)  # (true_class, pred_class) -> so luong, chi tinh misclassified
BATCH = 32
for i in range(0, len(all_imgs), BATCH):
    batch = all_imgs[i:i + BATCH]
    results = model.predict(source=[str(p) for p in batch], imgsz=640, conf=0.25, verbose=False)
    for img_path, res in zip(batch, results):
        with Image.open(img_path) as im:
            w, h = im.size
        gt_boxes = yolo_txt_to_xyxy(labels_dir / (img_path.stem + ".txt"), w, h)
        pred_xyxy = res.boxes.xyxy.cpu().numpy().tolist() if len(res.boxes) else []
        pred_cls = res.boxes.cls.cpu().numpy().astype(int).tolist() if len(res.boxes) else []
        preds = list(zip(pred_cls, pred_xyxy))

        used_pred = set()
        n_correct = n_miscls = n_missed = 0
        for gt_cid, *gt_box in gt_boxes:
            best_iou, best_j = 0.0, -1
            for j, (pc, pbox) in enumerate(preds):
                if j in used_pred:
                    continue
                iou = iou_xyxy(tuple(gt_box), tuple(pbox))
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= IOU_THRESH:
                used_pred.add(best_j)
                pc = preds[best_j][0]
                if pc == gt_cid:
                    n_correct += 1
                else:
                    n_miscls += 1
                    confusion_pairs[(TARGET_CLASSES[gt_cid], TARGET_CLASSES[pc])] += 1
            else:
                n_missed += 1
        n_extra = len(preds) - len(used_pred)

        per_image_records.append({
            "path": str(img_path), "filename": img_path.name,
            "n_gt": len(gt_boxes), "n_pred": len(preds),
            "correct": n_correct, "misclassified": n_miscls, "missed": n_missed, "extra": n_extra,
            "error_score": n_miscls + n_missed + n_extra,
        })
    print(f"  đã xử lý {min(i + BATCH, len(all_imgs))}/{len(all_imgs)} ảnh...")

img_df = pd.DataFrame(per_image_records)
print(f"\nTổng ảnh: {len(img_df)}")
print(f"Tổng box GT: {img_df['n_gt'].sum()} | correct: {img_df['correct'].sum()} | "
      f"misclassified: {img_df['misclassified'].sum()} | missed: {img_df['missed'].sum()} | "
      f"extra (báo thừa): {img_df['extra'].sum()}")

### Bước 2 — Ma trận nhầm lẫn (chỉ tính box định vị đúng nhưng sai class)

In [ ]:
confusion_df = pd.DataFrame(
    [{"true_class": t, "predicted_as": p, "count": c} for (t, p), c in confusion_pairs.items()]
).sort_values("count", ascending=False)
print(confusion_df.to_string(index=False))

pivot = confusion_df.pivot(index="true_class", columns="predicted_as", values="count").fillna(0).astype(int)
pivot = pivot.reindex(index=TARGET_CLASSES, columns=TARGET_CLASSES, fill_value=0)
print("\nMa trận (hàng = nhãn thật, cột = model đoán):")
print(pivot.to_string())

### Bước 3 — Xem trực quan top ảnh sai nhiều nhất
Vẽ **cả 2 loại box** trên cùng ảnh: nét đứt mảnh = ground truth thật, nét liền đậm = model dự đoán. Nếu box GT trông hợp lý (đúng màu quả) mà model vẫn đoán sai → nghiêng về domain shift. Nếu box GT trông đáng ngờ (nhãn sai rõ ràng) → nghiêng về lỗi nhãn.

In [ ]:
COLORS = ["lime", "orange", "red"]


def draw_gt_and_pred(ax, img_path, gt_boxes, pred_cls, pred_xyxy, pred_conf):
    img = Image.open(img_path)
    ax.imshow(img)
    for gt_cid, xmin, ymin, xmax, ymax in gt_boxes:
        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, linewidth=1.2,
                                  edgecolor=COLORS[gt_cid], facecolor="none", linestyle="--")
        ax.add_patch(rect)
        ax.text(xmin, min(ymax + 10, img.height - 2), f"GT:{TARGET_CLASSES[gt_cid][6:]}",
                color=COLORS[gt_cid], fontsize=6)
    for pc, (xmin, ymin, xmax, ymax), conf in zip(pred_cls, pred_xyxy, pred_conf):
        rect = patches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, linewidth=2,
                                  edgecolor=COLORS[pc], facecolor="none")
        ax.add_patch(rect)
        ax.text(xmin, max(ymin - 4, 0), f"P:{TARGET_CLASSES[pc][6:]} {conf:.2f}",
                color=COLORS[pc], fontsize=6, weight="bold")
    ax.axis("off")


def show_worst_images(df, n=6, title=""):
    top = df.sort_values("error_score", ascending=False).head(n)
    paths = [Path(p) for p in top["path"]]
    results = model.predict(source=[str(p) for p in paths], imgsz=640, conf=0.25, verbose=False)

    fig, axes = plt.subplots(1, len(paths), figsize=(4.2 * len(paths), 4.6))
    axes = [axes] if len(paths) == 1 else axes
    for ax, img_path, res, (_, row) in zip(axes, paths, results, top.iterrows()):
        with Image.open(img_path) as im:
            w, h = im.size
        gt_boxes = yolo_txt_to_xyxy(labels_dir / (img_path.stem + ".txt"), w, h)
        pred_xyxy = res.boxes.xyxy.cpu().numpy().tolist() if len(res.boxes) else []
        pred_cls = res.boxes.cls.cpu().numpy().astype(int).tolist() if len(res.boxes) else []
        pred_conf = res.boxes.conf.cpu().numpy().tolist() if len(res.boxes) else []
        draw_gt_and_pred(ax, img_path, gt_boxes, pred_cls, pred_xyxy, pred_conf)
        ax.set_title(f"{img_path.name}\nerr={row['error_score']} (miscls={row['misclassified']}, "
                     f"miss={row['missed']}, extra={row['extra']})", fontsize=6.5)

    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    save_path = Path("/kaggle/working") / f"worst_images_{title.replace(' ', '_')}.png"
    plt.savefig(save_path, dpi=90, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Đã lưu:", save_path)


show_worst_images(img_df, n=6, title="top6_loi_nhieu_nhat")

### Bước 4 — So sánh thống kê ảnh: trong miền vs. `openfield_bd`
Độ sáng (mean V) và độ bão hòa màu (mean S) trong không gian HSV — nếu khác biệt lớn, củng cố giả thuyết domain shift do ánh sáng/môi trường chụp (ngoài đồng có nắng tự nhiên khác hẳn đèn/nhà kính).

In [ ]:
def hsv_stats(img_path, resize=128):
    with Image.open(img_path) as im:
        im = im.convert("RGB").resize((resize, resize))
        hsv = im.convert("HSV")
        arr = np.asarray(hsv).astype(np.float32)
    return arr[..., 1].mean(), arr[..., 2].mean()  # saturation, brightness (0-255)


def sample_stats(img_dir, n=150):
    files = sorted(img_dir.glob("*"))
    sample = random.Random(42).sample(files, min(n, len(files))) if files else []
    rows = []
    for p in sample:
        try:
            s, v = hsv_stats(p)
            rows.append({"saturation": s, "brightness": v})
        except Exception:
            pass
    return pd.DataFrame(rows)


indomain_stats = sample_stats(DATASET_ROOT / "train" / "images", n=150)
outdomain_stats = sample_stats(OUTDOMAIN_DIR / "images", n=150)

summary = pd.DataFrame([
    {"group": "trong_mien (train)", "n": len(indomain_stats),
     "saturation_mean": round(indomain_stats["saturation"].mean(), 1),
     "brightness_mean": round(indomain_stats["brightness"].mean(), 1)},
    {"group": "openfield_bd (ngoai mien)", "n": len(outdomain_stats),
     "saturation_mean": round(outdomain_stats["saturation"].mean(), 1),
     "brightness_mean": round(outdomain_stats["brightness"].mean(), 1)},
])
print(summary.to_string(index=False))

sat_gap = abs(summary.loc[0, "saturation_mean"] - summary.loc[1, "saturation_mean"])
bri_gap = abs(summary.loc[0, "brightness_mean"] - summary.loc[1, "brightness_mean"])
print(f"\nChênh lệch saturation: {sat_gap:.1f} / 255 | Chênh lệch brightness: {bri_gap:.1f} / 255")
if sat_gap > 20 or bri_gap > 20:
    print("[GHI NHẬN] Chênh lệch đáng kể (>20/255) — ủng hộ giả thuyết domain shift do điều kiện chụp/ánh sáng khác nhau, "
          "không chỉ do nhãn.")
else:
    print("[GHI NHẬN] Chênh lệch không lớn — domain gap có thể đến từ yếu tố khác (bố cục, độ phân giải, giống cà chua...).")

### Bước 5 — Lưu kết quả điều tra

In [ ]:
WORKING = Path("/kaggle/working")
img_df.sort_values("error_score", ascending=False).to_csv(WORKING / "domain_gap_per_image.csv", index=False)
confusion_df.to_csv(WORKING / "domain_gap_confusion.csv", index=False)
summary.to_csv(WORKING / "domain_gap_image_stats.csv", index=False)

print("Đã lưu:")
print(" -", WORKING / "domain_gap_per_image.csv")
print(" -", WORKING / "domain_gap_confusion.csv")
print(" -", WORKING / "domain_gap_image_stats.csv")

print(f"\nTóm tắt: {img_df['error_score'].gt(0).sum()} / {len(img_df)} ảnh có ít nhất 1 lỗi "
      f"({100 * img_df['error_score'].gt(0).sum() / len(img_df):.1f}%).")

## Cách đọc kết quả
- **Ma trận nhầm lẫn (Bước 2)**: nếu lỗi tập trung mạnh vào 1-2 cặp class cụ thể (vd `fruit_turning` → `fruit_green_unripe`) một cách hệ thống, đó là dấu hiệu ranh giới màu giữa các giai đoạn chín trong điều kiện ánh sáng ngoài đồng khác với ranh giới model học được từ ảnh trong nhà kính — ủng hộ domain shift hơn là lỗi nhãn ngẫu nhiên.
- **Ảnh lỗi nhiều nhất (Bước 3)**: đây là bước quan trọng nhất để tự kết luận. Nếu box GT (nét đứt) trông khớp đúng với màu quả thật trong ảnh mà model vẫn đoán khác (box nét liền) → model đang gặp khó vì domain shift, **không phải lỗi nhãn**. Chỉ khi box GT tự nó trông sai (vd quả rõ ràng đỏ nhưng gắn nhãn `fruit_green_unripe`) mới là lỗi nhãn giống trường hợp `leaf_mold`.
- **Thống kê ảnh (Bước 4)**: chênh lệch saturation/brightness lớn là bằng chứng định lượng độc lập, không phụ thuộc vào việc đọc nhãn đúng hay sai.

## Nếu kết luận là domain shift thật (nhiều khả năng, vì class mapping đã xác nhận đúng)
- Không có "fix" bằng cách sửa nhãn — cần **thu thập ảnh thật từ camera IMX179** trong đúng môi trường nhà kính triển khai, theo đúng khuyến nghị của tài liệu dự án.
- Trong lúc chờ dữ liệu thật: cân nhắc thêm color-jitter/augmentation ánh sáng mạnh hơn khi train phiên bản chính thức để tăng khả năng tổng quát hóa, dù không thay thế được dữ liệu thật.
- Tiếp tục báo cáo `test_outdomain_openfield` riêng biệt trong mọi lần đánh giá — không gộp vào số liệu chính để tránh đánh giá sai lạc quan.

## Nếu phát hiện một số ảnh nhãn sai cụ thể
- Áp dụng đúng mẫu đã làm ở nhánh bệnh lá: ghi lại danh sách, cân nhắc loại trừ ở lần build kế tiếp, train lại và so sánh.